In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

# Thay đổi tên thư mục cho đúng với tên bạn đặt trên Drive
data_folder = '/content/drive/MyDrive/Dataset cuối kỳ'

# Liệt kê các file trong thư mục
print(os.listdir(data_folder))

['olist_customers_dataset.csv', 'product_category_name_translation.csv', 'olist_sellers_dataset.csv', 'App.py', 'olist_order_reviews_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_items_dataset.csv', 'olist_orders_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_products_dataset.csv']


In [ ]:
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

# --- CẤU HÌNH ĐƯỜNG DẪN ---
# Đường dẫn bạn đã cung cấp
DATA_FOLDER = Path('/content/drive/MyDrive/Dataset cuối kỳ')

print(f"Đang kiểm tra đường dẫn: {DATA_FOLDER}")
if not DATA_FOLDER.exists():
    raise FileNotFoundError(f"Không tìm thấy thư mục tại: {DATA_FOLDER}. Vui lòng kiểm tra lại tên thư mục trên Drive.")

# --- HÀM LOAD VÀ XỬ LÝ DỮ LIỆU THẬT ---
def load_olist_data():
    try:
        print("Đang tải các file CSV...")
        # Load từng file
        orders = pd.read_csv(DATA_FOLDER / 'olist_orders_dataset.csv')
        items = pd.read_csv(DATA_FOLDER / 'olist_order_items_dataset.csv')
        products = pd.read_csv(DATA_FOLDER / 'olist_products_dataset.csv')
        payments = pd.read_csv(DATA_FOLDER / 'olist_order_payments_dataset.csv')
        reviews = pd.read_csv(DATA_FOLDER / 'olist_order_reviews_dataset.csv')

        print("Đang gộp dữ liệu (Merge)...")
        # Merge các bảng
        df = orders.merge(items, on='order_id', how='left')
        df = df.merge(products, on='product_id', how='left')
        df = df.merge(payments, on='order_id', how='left')
        df = df.merge(reviews, on='order_id', how='left')

        # Xử lý ngày tháng
        df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'], errors='coerce')
        df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'], errors='coerce')

        # Tính số ngày giao hàng
        df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

        # Làm sạch dữ liệu số
        df['price'] = pd.to_numeric(df['price'], errors='coerce')
        df['freight_value'] = pd.to_numeric(df['freight_value'], errors='coerce')
        df['review_score'] = pd.to_numeric(df['review_score'], errors='coerce')

        print(f"✅ Thành công! Đã load {len(df)} bản ghi.")
        return df

    except Exception as e:
        print(f"❌ Lỗi khi tải dữ liệu: {str(e)}")
        return None

# Load dữ liệu toàn cục
df_olist = load_olist_data()

# --- CÁC HÀM XỬ LÝ CHO TỪNG TAB ---

def get_dashboard_stats():
    if df_olist is None:
        return None, pd.DataFrame({"Error": ["No data"]})

    # Metrics
    total_orders = df_olist['order_id'].nunique()
    total_revenue = df_olist['price'].sum()
    avg_review = df_olist['review_score'].mean()

    # Biểu đồ doanh thu theo tháng
    df_monthly = df_olist.groupby(df_olist['order_purchase_timestamp'].dt.to_period('M')).agg({
        'price': 'sum',
        'order_id': 'count'
    }).reset_index()
    df_monthly.columns = ['Month', 'Revenue', 'Orders']
    df_monthly['Month'] = df_monthly['Month'].astype(str)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Revenue
    axes[0, 0].plot(df_monthly['Month'], df_monthly['Revenue'], marker='o', color='green', linewidth=2)
    axes[0, 0].set_title('Doanh thu theo tháng (BRL)', fontsize=12, fontweight='bold')
    axes[0, 0].tick_params(axis='x', rotation=45)

    # 2. Số đơn hàng
    axes[0, 1].bar(df_monthly['Month'], df_monthly['Orders'], color='blue', alpha=0.7)
    axes[0, 1].set_title('Số đơn hàng theo tháng', fontsize=12, fontweight='bold')
    axes[0, 1].tick_params(axis='x', rotation=45)

    # 3. Phân bố điểm đánh giá
    review_counts = df_olist['review_score'].value_counts().sort_index()
    axes[1, 0].bar(review_counts.index, review_counts.values, color='orange')
    axes[1, 0].set_title('Phân bố điểm đánh giá (Review Score)', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Score')

    # 4. Top danh mục sản phẩm
    if 'product_category_name' in df_olist.columns:
        top_cats = df_olist['product_category_name'].value_counts().head(8)
        axes[1, 1].barh(top_cats.index, top_cats.values, color='purple')
        axes[1, 1].set_title('Top 8 Danh mục sản phẩm bán chạy', fontsize=12, fontweight='bold')

    plt.tight_layout()

    stats_df = pd.DataFrame({
        'Chỉ số': ['Tổng đơn hàng', 'Tổng doanh thu (BRL)', 'Điểm đánh giá TB'],
        'Giá trị': [f"{total_orders:,}", f"{total_revenue:,.0f}", f"{avg_review:.2f}"]
    })

    return fig, stats_df

def run_segmentation():
    if df_olist is None:
        return "Lỗi: Chưa có dữ liệu", None

    # Chuẩn bị dữ liệu khách hàng
    cust_stats = df_olist.groupby('customer_id').agg({
        'order_id': 'count',
        'price': 'sum',
        'review_score': 'mean'
    }).reset_index()
    cust_stats.columns = ['customer_id', 'order_count', 'total_spent', 'avg_review']

    # KMeans Clustering
    features = ['order_count', 'total_spent']
    X = cust_stats[features].fillna(0)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    cust_stats['cluster'] = kmeans.fit_predict(X_scaled)

    # Thống kê cụm
    cluster_summary = cust_stats.groupby('cluster').agg({
        'customer_id': 'count',
        'order_count': 'mean',
        'total_spent': 'mean',
        'avg_review': 'mean'
    }).round(2)
    cluster_summary.columns = ['Số KH', 'TB Đơn/KH', 'TB Chi Tiêu', 'TB Review']

    # Vẽ biểu đồ
    fig, ax = plt.subplots(figsize=(10, 6))
    scatter = ax.scatter(cust_stats['order_count'], cust_stats['total_spent'],
                         c=cust_stats['cluster'], cmap='viridis', alpha=0.6, s=20)
    ax.set_title('Phân khúc Khách hàng (K-Means Clustering)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Số lần mua hàng')
    ax.set_ylabel('Tổng chi tiêu (BRL)')
    plt.colorbar(scatter, label='Cluster ID')
    plt.tight_layout()

    text_result = f"Đã phân khúc {len(cust_stats)} khách hàng vào 4 nhóm.\n\nChi tiết các nhóm:\n{cluster_summary.to_string()}"

    return text_result, fig

def get_recommendations(customer_id=None):
    if df_olist is None:
        return "Lỗi dữ liệu", pd.DataFrame()

    # Logic đơn giản: Gợi ý Top sản phẩm bán chạy nhất toàn hệ thống
    # (Vì không có mô hình ALS phức tạp trong demo này, ta dùng thống kê tần suất)
    top_products = df_olist.groupby(['product_id', 'product_category_name']).agg({
        'price': ['count', 'sum']
    }).reset_index()
    top_products.columns = ['product_id', 'category', 'sold_count', 'revenue']
    top_10 = top_products.nlargest(10, 'sold_count')

    msg = f"Top 10 sản phẩm bán chạy nhất (Dựa trên {len(df_olist)} đơn hàng thực tế)"
    if customer_id:
        msg += f" - Tìm kiếm cho KH: {customer_id}"

    return msg, top_10[['product_id', 'category', 'sold_count']]

def get_association_rules():
    if df_olist is None:
        return pd.DataFrame()

    # Giả lập Association Rules dựa trên đồng xuất hiện danh mục
    # Nhóm các sản phẩm trong cùng 1 đơn hàng
    order_cats = df_olist.groupby('order_id')['product_category_name'].apply(list).reset_index()

    rules = []
    categories = df_olist['product_category_name'].dropna().unique()[:20] # Lấy 20 cat phổ biến để tính nhanh

    for i, cat1 in enumerate(categories):
        for cat2 in categories[i+1:]:
            count = 0
            for cats in order_cats['product_category_name']:
                if cat1 in cats and cat2 in cats:
                    count += 1
            if count > 10: # Support tối thiểu
                rules.append({'Danh mục 1': cat1, 'Danh mục 2': cat2, 'Số lần mua cùng': count})

    rules_df = pd.DataFrame(rules)
    if len(rules_df) > 0:
        return rules_df.sort_values('Số lần mua cùng', ascending=False).head(10)
    return pd.DataFrame({'Info': ['Không tìm thấy quy tắc nào đủ điều kiện support']})

def predict_review(price, freight, days):
    if df_olist is None:
        return "Chưa có dữ liệu"

    # Sử dụng thống kê thực tế để dự đoán (Rule-based đơn giản hóa từ dữ liệu thật)
    # Tính trung bình thực tế
    avg_price = df_olist['price'].mean()
    avg_freight = df_olist['freight_value'].mean()
    avg_days = df_olist['delivery_days'].dropna().mean()
    base_score = df_olist['review_score'].mean()

    score = base_score

    # Điều chỉnh dựa trên độ lệch so với trung bình
    if days > avg_days * 1.5:
        score -= 1.5
    elif days < avg_days * 0.8:
        score += 0.5

    if freight > avg_freight * 1.5:
        score -= 0.5

    # Giới hạn 1-5
    score = max(1.0, min(5.0, score))

    result_text = f"""
    📊 DỰ ĐOÁN DỰA TRÊN DỮ LIỆU THỰC TẾ OLIST

    Thông tin đầu vào:
    - Giá: {price:.2f} BRL (TB hệ thống: {avg_price:.2f})
    - Phí vận chuyển: {freight:.2f} BRL (TB hệ thống: {avg_freight:.2f})
    - Thời gian giao: {days:.1f} ngày (TB hệ thống: {avg_days:.1f})

    🔮 Điểm đánh giá dự đoán: {score:.1f} / 5.0

    💡 Nhận xét:
    {'Giao hàng chậm hơn mức trung bình, khách có thể không hài lòng.' if days > avg_days else 'Thời gian giao hàng tốt, khả năng cao được đánh giá cao.'}
    """
    return result_text

# --- XÂY DỰNG GIAO DIỆN GRADIO ---

with gr.Blocks(theme=gr.themes.Soft(), title="Olist E-Commerce Analytics") as demo:
    gr.Markdown("""
    # 🇧🇷 HỆ THỐNG PHÂN TÍCH DỮ LIỆU OLIST E-COMMERCE
    ### Phân tích hành vi khách hàng Brazil từ Dataset thực tế (~100k đơn hàng)
    """)

    with gr.Tab("🏠 Dashboard Tổng Quan"):
        gr.Markdown("### Thống kê hiệu suất kinh doanh")
        with gr.Row():
            metric_orders = gr.Number(label="Tổng đơn hàng", value=0)
            metric_revenue = gr.Number(label="Tổng doanh thu (BRL)", value=0)
            metric_review = gr.Number(label="Điểm Review TB", value=0)

        dash_plot = gr.Plot(label="Biểu đồ phân tích")
        dash_table = gr.Dataframe(label="Tóm tắt chỉ số")

        def update_dash():
            if df_olist is not None:
                fig, stats_df = get_dashboard_stats()
                total_orders = df_olist['order_id'].nunique()
                total_rev = df_olist['price'].sum()
                avg_rev = df_olist['review_score'].mean()
                return total_orders, total_rev, avg_rev, fig, stats_df
            return 0, 0, 0, None, pd.DataFrame()

        demo.load(update_dash, inputs=None, outputs=[metric_orders, metric_revenue, metric_review, dash_plot, dash_table])

    with gr.Tab("👥 Phân Khúc Khách Hàng"):
        gr.Markdown("### Phân nhóm khách hàng sử dụng K-Means Clustering")
        seg_btn = gr.Button("Chạy phân khúc", variant="primary")
        seg_text = gr.Textbox(label="Kết quả phân tích", lines=10)
        seg_plot = gr.Plot(label="Biểu đồ phân cụm")

        seg_btn.click(run_segmentation, inputs=None, outputs=[seg_text, seg_plot])

    with gr.Tab("💡 Hệ Thống Khuyến Nghị"):
        gr.Markdown("### Gợi ý sản phẩm bán chạy (Top 10)")
        cust_input = gr.Textbox(label="Nhập Customer ID (Tùy chọn)", placeholder="VD: 12345...")
        rec_btn = gr.Button("Tìm gợi ý", variant="primary")
        rec_msg = gr.Textbox(label="Thông báo")
        rec_table = gr.Dataframe(label="Danh sách sản phẩm")

        rec_btn.click(get_recommendations, inputs=cust_input, outputs=[rec_msg, rec_table])

    with gr.Tab("🛒 Xu Hướng Mua Sắm"):
        gr.Markdown("### Luật kết hợp sản phẩm (Association Rules)")
        rule_btn = gr.Button("Phân tích xu hướng", variant="primary")
        rule_table = gr.Dataframe(label="Các cặp danh mục thường mua cùng")

        rule_btn.click(get_association_rules, inputs=None, outputs=rule_table)

    with gr.Tab("🔮 Dự Đoán Review"):
        gr.Markdown("### Dự đoán điểm đánh giá dựa trên thông tin đơn hàng")
        with gr.Row():
            p_price = gr.Number(label="Giá sản phẩm (BRL)", value=100)
            p_freight = gr.Number(label="Phí vận chuyển (BRL)", value=20)
            p_days = gr.Number(label="Số ngày giao hàng", value=15)

        pred_btn = gr.Button("Dự đoán ngay", variant="primary")
        pred_result = gr.Textbox(label="Kết quả dự đoán", lines=10)

        pred_btn.click(predict_review, inputs=[p_price, p_freight, p_days], outputs=pred_result)

if __name__ == "__main__":
    # Chia sẻ công khai để dễ chụp màn hình báo cáo
    demo.launch(share=True)

Đang kiểm tra đường dẫn: /content/drive/MyDrive/Dataset cuối kỳ
Đang tải các file CSV...
Đang gộp dữ liệu (Merge)...
✅ Thành công! Đã load 119143 bản ghi.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ee6d5b25eddfb48fda.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
